In [ ]:
import pycountry
import pandas as pd
import numpy as np

# 모든 국가 정보 가져오기
countries = list(pycountry.countries)

# 국가명과 ISO 코드 출력
for country in countries[:5]: #상위 5개만 출력
    print(country.name, country.alpha_2, country.alpha_3, country.numeric)

In [ ]:
# 한글 국가명 붙이기
import requests
from bs4 import BeautifulSoup

# 위키백과 URL
url = "https://ko.wikipedia.org/wiki/ISO_3166-1"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# 테이블 찾기
table = soup.find("table", {"class": "wikitable"})

# 국가명 데이터 추출 (Numeric 코드 기준)
country_name_ko = {}

# 테이블 행 반복
for row in table.find_all("tr")[1:]:  # 첫 번째 행은 헤더라서 제외
    cols = row.find_all("td")
    
    if len(cols) >= 4:
        # 한글 국가명은 첫 번째 <td> 안에 텍스트로 들어 있음
        ko_name = cols[0].text.strip()
        
        numeric_code = cols[1].text.strip()  # 숫자 코드
        iso3_code = cols[2].text.strip()  # ISO 3 코드
        iso2_code = cols[3].text.strip()  # ISO 2 코드

        # Numeric 코드 기준으로 한글 국가명 저장
        country_name_ko[numeric_code] = {
            'ko_name': ko_name,
            'iso3': iso3_code,
            'iso2': iso2_code
        }

# 결과 출력
for numeric_code, country_info in country_name_ko.items():
    print(f"Numeric Code: {numeric_code}, 국가명: {country_info['ko_name']}, ISO3: {country_info['iso3']}, ISO2: {country_info['iso2']}")


In [ ]:
# country_name_ko 데이터를 데이터프레임으로 변환
df = pd.DataFrame.from_dict(country_name_ko, orient='index')

# 'numeric_code'를 별도로 컬럼으로 추가
df['Numeric Code'] = df.index

# 인덱스를 0부터 시작하도록 설정
df = df.reset_index(drop=True)

# 컬럼 이름 지정
df.columns = ['국가명', 'ISO3', 'ISO2', 'Numeric Code']

# 결과 출력
df

In [ ]:
import pandas as pd
import time
from pytrends.request import TrendReq
pd.set_option('future.no_silent_downcasting', True)

# pytrends 설정
pytrends = TrendReq(hl='ko', tz=540)

# 국가 목록을 리스트로 변환
kw_list = df['국가명'].to_list()

# 날짜 범위 계산 (최소 날짜와 최대 날짜 계산)
start_date = pd.to_datetime('2021-02-01')  
end_date = pd.to_datetime('2025-02-26')    
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

# 5개씩 요청 후 국가별로 데이터 저장
for i in range(0, len(kw_list), 5):
    try:
        # 키워드 목록을 설정하여 데이터 요청
        pytrends.build_payload(kw_list[i:i+5], cat=67, timeframe='today 5-y', geo='KR', gprop='')

        # 데이터 가져오기 (시간대별 관심도)
        df_interest = pytrends.interest_over_time()

        # 빈 데이터프레임이 아니면 처리
        if not df_interest.empty:
           
            # NaN값을 0으로 채우기
            df_interest = df_interest.fillna(0)

            # 국가별로 개별 데이터프레임 생성
            for col in df_interest.columns:
                if col != 'isPartial':  # 'isPartial' 컬럼 제외
                    globals()[f"{col}_data"] = df_interest[[col]]
        
        # 요청 간 대기 시간을 추가하여 rate limit을 피하기 위해 잠시 대기
        time.sleep(120)  # 대기 시간 60초 설정

    except Exception as e:
        print(f"Error with the request for {kw_list[i:i+5]}: {e}")
        time.sleep(180)  # 오류 발생 시 더 긴 대기 (180초) 후 재시도

# 결과 출력 (각 국가별 데이터프레임 출력)
for country in kw_list:
    var_name = f"{country}_data"
    if var_name in globals():
        print(f"--- {var_name} ---")
        print(globals()[var_name])


In [ ]:
# 국가별 데이터프레임을 저장할 딕셔너리 초기화
monthly_data_dict = {}

# 각 국가에 대해 데이터를 월별로 집계
for country in kw_list:
    var_name = f"{country}_data"
    if var_name in globals():
        # 국가별 데이터프레임 가져오기
        df_interest = globals()[var_name].copy()  # 원본 유지

        # 'date' 컬럼을 datetime 형식으로 변환
        df_interest['date'] = pd.to_datetime(df_interest.index)

        # 연도와 월을 기준으로 그룹화
        df_interest['year_month'] = df_interest['date'].dt.to_period('M')

        # 숫자형 데이터만 선택하여 합산
        numeric_cols = df_interest.select_dtypes(include=['number']).columns
        monthly_data = df_interest.groupby('year_month')[numeric_cols].mean()

        # 딕셔너리에 국가별 데이터 저장
        monthly_data_dict[country] = monthly_data

        # 결과 출력 (옵션)
        print(f"--- {country} 월별 데이터 ---")
        print(monthly_data)


In [ ]:
# 국가별 데이터를 concat하여 하나의 데이터프레임으로 합침
final_df = pd.concat(monthly_data_dict.values(), axis=1)

# 'year_month'를 인덱스로 설정 (기존 인덱스 유지)
final_df.index.name = 'year_month'

# 컬럼 중복 방지를 위해 기존 컬럼명을 국가명으로 대체
final_df.columns = monthly_data_dict.keys()

# 결과 출력 (옵션)
print("합쳐진 데이터프레임:")
print(final_df)


In [ ]:
final_df

In [ ]:
# 모든 값이 0인 컬럼 제거
final_df_1 = final_df.loc[:, (final_df != 0).any(axis=0)]

# 50% 이상이 0인 컬럼 제거
final_df_1 = final_df_1.loc[:, final_df_1.apply(lambda x: (x == 0).mean() < 0.5, axis=0)]

In [ ]:
final_df_1

In [ ]:
final_df_1.columns

## 검색량 데이터 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한글 폰트 설정
plt.rc('font', family='AppleGothic')  

columns = final_df_1.columns

# x축이 PeriodIndex라면 datetime으로 변환
if isinstance(final_df_1.index, pd.PeriodIndex):
    final_df_1 = final_df_1.copy()
    final_df_1.index = final_df_1.index.to_timestamp()

rows, cols = 8, 4  
fig, axes = plt.subplots(rows, cols, figsize=(18, 18))
axes = axes.flatten()  # 1D 배열로 변환

# 각 컬럼에 대해 그래프 그리기
for i, col in enumerate(columns):
    axes[i].plot(final_df_1.index, final_df_1[col])
    axes[i].set_title(col, fontsize=10)  # 한글 폰트 적용
    axes[i].tick_params(axis='x', rotation=45)

# 남은 빈 그래프 삭제
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
# 이미지 저장
plt.savefig('country_graph.png') 
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
from statsmodels.tsa.seasonal import seasonal_decompose

# 한글 폰트 설정 
matplotlib.rcParams['font.family'] = 'AppleGothic'  
matplotlib.rcParams['axes.unicode_minus'] = False 

final_df_1.index = pd.to_datetime(final_df_1.index)  

# 모든 컬럼에 대해 시계열 분해
for country in final_df_1.columns:
    ts = final_df_1[country]  # 해당 국가의 검색량 데이터 추출
    
    # 시계열 분해 (1년 주기 가정)
    result = seasonal_decompose(ts, model='additive', period=12)
    
    # 결과 시각화
    fig, axes = plt.subplots(4, 1, figsize=(10, 8))
    result.observed.plot(ax=axes[0], title=f"{country} - Observed")
    result.trend.plot(ax=axes[1], title=f"{country} - Trend")
    result.seasonal.plot(ax=axes[2], title=f"{country} - Seasonal")
    result.resid.plot(ax=axes[3], title=f"{country} - Residual")
    
    plt.tight_layout()
    plt.show()